# CodeTMS initial data visualizations

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from matplotlib import pyplot as plt

## Settings

In [ ]:
input_version = 1
output_version = 1
save_plots = True

In [ ]:
# make the output directory
if save_plots:
    path = os.path.realpath("__file__")[:-len("__file__")]
    output_path = os.path.join(path, "visualizations-{}".format(output_version))
    try:
        os.mkdir(output_path)
    except OSError as error:
        print(error)

## Read in the data

In [ ]:
df = pd.read_csv("processed-data-{}/tms-functional-data-v{}.csv".format(input_version, input_version), converters={"id" : str})

df.rename(columns={"adjusted_stimulus" : "stimulus"}, inplace=True)

display(df.head())

In [ ]:
df["domain"].value_counts()

In [ ]:
# see how they're not even?

In [ ]:
# make a dataframe where the "individuals" are the question prompts rather than the people
# each row is a question prompt ("stimulus"), and stores its category (domain), rate at
# which people got it correct (across all TMS treatment conditions), number of times it was
# seen, and average response time (across all TMS treatment conditions)
stimuli = df.groupby("stimulus").agg({"domain" : "first", "correct" : [np.mean, len], "response_time" : np.mean, "test_version" : "first"}).reset_index()
stimuli.columns = ["stimulus", "domain", "correct_rate", "count", "avg_time", "test_version"]
stimuli

In [ ]:
stimuli["count"].value_counts()

So 33 stimuli have been observed 14 times, 100 stimuli have been observed 15 times, and 50 stimuli have been observed 16 times.

In [ ]:
df.groupby("test_version")["id"].nunique()

The above cell gives the number of times that each test version was used.

## Visualization

If your data are not anonymized, change these to 

In [ ]:
# basic
colors = {"A" : "red", "B" : "orange", "C" : "SeaGreen", "D" : "CornflowerBlue", "E" : "DarkMagenta"}

# pastel 1
#colors = {"list/array" : "LightCoral", "tree" : "SandyBrown", "shepard-metzler" : "LightGreen", "PSVT:R II" : "LightSkyBlue", "code" : "Violet"}

# pastel 2
#colors = {"list/array" : "#ffb3ba", "tree" : "#ffdfba", "shepard-metzler" : "#ffffba", "PSVT:R II" : "#baffc9", "code" : "#bae1ff"}

# grayscale
#colors = {"list/array" : "0.8", "tree" : "0.6", "shepard-metzler" : "0.4", "PSVT:R II" : "0.2", "code" : "0.0"}

# neon
#colors = {"list/array" : "DeepPink", "tree" : "Gold", "shepard-metzler" : "Lime", "PSVT:R II" : "cyan", "code" : "BlueViolet"}


In [ ]:
plt.figure(figsize=(12, 3))
for cat, color in colors.items():
    xs = stimuli[stimuli["domain"] == cat]["stimulus"]
    ys = stimuli[stimuli["domain"] == cat]["correct_rate"]
    plt.scatter(xs, ys, s=25, c=color, label=cat)

plt.xlabel("question number")
plt.ylabel("correct rate")
plt.title("correct rates of each question, colored by question type")
plt.legend()

if save_plots:
    plt.savefig(os.path.join(output_path, "corr-rate_question-num_domain.png"), bbox_inches="tight")

plt.show()

In [ ]:
plt.figure(figsize=(12, 3))
for cat, color in colors.items():
    xs = stimuli[stimuli["domain"] == cat]["stimulus"]
    ys = stimuli[stimuli["domain"] == cat]["avg_time"]
    plt.scatter(xs, ys, s=25, c=color, label=cat)

plt.xlabel("question number")
plt.ylabel("avg response time")
plt.title("avg response times of each question, colored by question type")
plt.legend()

if save_plots:
    plt.savefig(os.path.join(output_path, "resp-time_question-num_domain.png"), bbox_inches="tight")
    
plt.show()

In [ ]:
plt.figure(figsize=(12, 3))
for cat, color in colors.items():
    xs = stimuli[stimuli["domain"] == cat]["avg_time"]
    ys = stimuli[stimuli["domain"] == cat]["correct_rate"]
    plt.scatter(xs, ys, s=25, c=color, alpha=1, label=cat)

plt.xlabel("average response time")
plt.ylabel("correct rate")
plt.title("correct rate vs average response time of each question")
plt.legend()

if save_plots:
    plt.savefig(os.path.join(output_path, "corr-rate_resp-time_domain.png"), bbox_inches="tight")
    
plt.show()

In [ ]:
colors = {"X" : "red", "Y" : "green", "Z" : "blue"}

In [ ]:
#bounds = [0, 31, 61, 91, 121, 151, 208, 215]
bounds = stimuli.groupby("domain").first()["stimulus"].tolist() + [max(stimuli["stimulus"])]
bounds.sort()
bounds[0] = 0

In [ ]:
# start off the plot
plt.figure(figsize=(12, 3))

# plot all the points for each experimental condition
for condition, color in colors.items():
    stratified_stimuli = df[df["condition"] == condition].groupby("stimulus").agg({"domain" : "first", "correct" : [np.mean, len], "response_time" : np.mean}).reset_index()
    stratified_stimuli.columns = ["stimulus", "domain", "correct_rate", "count", "avg_time"]
    
    
    xs = stratified_stimuli["stimulus"]
    ys = stratified_stimuli["correct_rate"]
    plt.scatter(xs, ys, s=5, c=color, alpha=0.5, label=condition)

# section off the different categories of questions
for k in range(0, len(bounds), 2):
    plt.axvspan(bounds[k], bounds[k+1], alpha=0.3, color="0.5")

# score up the empty block of indices
#plt.axvspan(151, 200, alpha=0.5, facecolor="1.0", edgecolor="0.0", hatch="x")

# set the labels
plt.xlabel("question number")
plt.ylabel("correct rate")
plt.title("correct rates of each question, separated by experimental condition")
plt.legend(loc=4)

if save_plots:
    plt.savefig(os.path.join(output_path, "corr-rate_question-num_ condition.png"), bbox_inches="tight")
    
plt.show()

The stimulus categories go:
<p style="text-align: right;">
    <i> A </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp;&emsp; 
    <i> B </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; 
    <i> C </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &ensp;
    <i> D </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp;
    <i> E </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp;
</p>

In [ ]:
# start off the plot
plt.figure(figsize=(12, 3))

# plot all the points for each experimental condition
for condition, color in colors.items():
    stratified_stimuli = df[df["condition"] == condition].groupby("stimulus").agg({"domain" : "first", "correct" : [np.mean, len], "response_time" : np.mean}).reset_index()
    stratified_stimuli.columns = ["stimulus", "domain", "correct_rate", "count", "avg_time"]
    
    xs = stratified_stimuli["stimulus"]
    ys = stratified_stimuli["avg_time"]
    plt.scatter(xs, ys, s=5, c=color, label=condition)

# section off the different categories of questions
for k in range(0, len(bounds), 2):
    plt.axvspan(bounds[k], bounds[k+1], alpha=0.3, color="0.5")

# score up the empty block of indices
#plt.axvspan(151, 200, alpha=0.5, facecolor="1.0", edgecolor="0.0", hatch="x")

# set the labels
plt.xlabel("question number")
plt.ylabel("avg response time")
plt.title("avg response times of each question, separated by experimental condition")
plt.legend(loc=1)

if save_plots:
    plt.savefig(os.path.join(output_path, "resp-time_question-num_condition.png"), bbox_inches="tight")

plt.show()

Same as above, the stimulus categories go:
<p style="text-align: right;">
    <i> A </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp;&emsp; 
    <i> B </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; 
    <i> C </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &ensp;
    <i> D </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp;
    <i> E </i> &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp; &emsp;
</p>

In [ ]:
plt.figure(figsize=(12, 3))
for condition, color in colors.items():
    stratified_stimuli = df[df["condition"] == condition].groupby("stimulus").agg({"domain" : "first", "correct" : [np.mean, len], "response_time" : np.mean}).reset_index()
    stratified_stimuli.columns = ["stimulus", "domain", "correct_rate", "count", "avg_time"]
    
    xs = stratified_stimuli["avg_time"]
    ys = stratified_stimuli["correct_rate"]
    plt.scatter(xs, ys, s=5, c=color, label=condition)

plt.xlabel("average response time")
plt.ylabel("correct rate")
plt.title("correct rate vs average response time of each question, separated by experimental condition")
plt.legend()

if save_plots:
    plt.savefig(os.path.join(output_path, "corr-rate_resp-time_condition.png"), bbox_inches="tight")

plt.show()

This graph looks pretty gross because all the points are plotted on top of each other.

## Examine outlier stimuli
From Figure 1, we can see a few outlier questions with very poor performance, leading us to wonder if they might have been coded incorrectly. Here, we pull the indices for those questions so we can double check.

In [ ]:
thresh = 0.5 # from looking at the models

In [ ]:
stimuli[stimuli["correct_rate"] < thresh]

In [ ]:
for stimulus in stimuli[stimuli["correct_rate"] < thresh]["stimulus"].tolist():
    correct_response = df[df["stimulus"] == stimulus]["correct_response"].iloc[0]
    original_index = df[df["stimulus"] == stimulus]["original_stimulus"].iloc[0]
    print("Stimulus #{} (originally #{}): answer choice {}".format(stimulus, original_index, correct_response))